# Chrome 小恐龍自動遊玩

在 VS Code 的 Notebook 右上角選擇 `iem_python` 核心，並在該環境安裝 PyAutoGUI（`conda install -c conda-forge pyautogui`）。先在 Chrome 開啟 `chrome://dino/`，保持遊戲視窗可見。

1. 在主螢幕最大化 Chrome、將瀏覽器縮放設為 100%，開啟 `chrome://dino/`。手動按向上鍵開始，等待出現 `GAME OVER`；此時遊戲已放大到固定位置。
2. 執行下方儲存格，在 8 秒內用 Alt + Tab 切回 Chrome（不要點擊遊戲），將滑鼠移到**恐龍頭部右邊、靠近水平地面線**的位置並保持不動；程式會自動校準地面線。
3. 程式會點擊遊戲重新開始，以截圖顏色判斷障礙物距離；地面障礙物進入起跳範圍時按向上鍵，飛行障礙物靠近時按住向下鍵。執行期間不要移動、縮放或遮住遊戲視窗。

執行時會顯示即時截圖預覽：紅框是最近的障礙物，黃框是其他障礙物，藍線是目前的起跳範圍邊界。預覽視窗會自動避開遊戲偵測區；按預覽右上角的「關閉」可關閉預覽，遊戲程式仍會繼續。

按 Notebook 的停止按鈕，或將滑鼠移到螢幕左上角可停止。若跳得太晚，可增加 `JUMP_DISTANCE`；若跳得太早，可減少它。畫面擷取與按鍵仍有延遲，不能保證永遠不會死亡。

In [ ]:
import ctypes
import sys
import time
import tkinter as tk
import pyautogui as pyg
from PIL import Image, ImageDraw, ImageTk

# 起跳距離（換算後的畫面像素）：增加會提早跳，減少會更靠近再跳。
JUMP_DISTANCE = 70
JUMP_COOLDOWN = 0.60
COLOR_DIFFERENCE = 60
SCAN_WIDTH = 330
SCAN_HEIGHT = 73
SHOW_PREVIEW = True  # 改為 False 可隱藏截圖預覽。

def foreground_window():
    """取得 Windows 目前接收鍵盤輸入的視窗。"""
    if sys.platform != 'win32':
        return None
    user32 = ctypes.windll.user32
    user32.GetForegroundWindow.restype = ctypes.c_void_p
    return user32.GetForegroundWindow()

def prevent_preview_activation(root):
    """避免點選預覽時將 Chrome 的鍵盤焦點搶走。"""
    if sys.platform != 'win32':
        return
    user32 = ctypes.windll.user32
    user32.GetAncestor.argtypes = (ctypes.c_void_p, ctypes.c_uint)
    user32.GetAncestor.restype = ctypes.c_void_p
    user32.GetWindowLongW.argtypes = (ctypes.c_void_p, ctypes.c_int)
    user32.GetWindowLongW.restype = ctypes.c_long
    user32.SetWindowLongW.argtypes = (ctypes.c_void_p, ctypes.c_int, ctypes.c_long)
    user32.SetWindowLongW.restype = ctypes.c_long
    hwnd = user32.GetAncestor(root.winfo_id(), 2)
    if hwnd:
        style = user32.GetWindowLongW(hwnd, -20)
        user32.SetWindowLongW(hwnd, -20, style | 0x08000000)  # WS_EX_NOACTIVATE

def contrast_threshold(background):
    # 日夜漸變時降低門檻，讓較淡的障礙物仍能被偵測。
    return min(COLOR_DIFFERENCE, max(16, abs(background - 128)))

def find_ground_line(image, selected_row):
    """在滑鼠附近尋找長度足夠的水平地面線。"""
    gray = image.convert('L')
    histogram = gray.histogram()
    background = max(range(256), key=histogram.__getitem__)
    threshold = contrast_threshold(background)
    pixels = gray.load()
    candidates = []
    for row in range(gray.height):
        foreground = sum(abs(pixels[column, row] - background) >= threshold
                         for column in range(gray.width))
        if foreground >= gray.width * 0.55:
            candidates.append(row)
    return min(candidates, key=lambda row: abs(row - selected_row)) if candidates else None

def find_obstacles(image):
    """找出與畫面背景有足夠色差、並接近恐龍高度的物體。"""
    gray = image.convert('L')
    histogram = gray.histogram()
    background = max(range(256), key=histogram.__getitem__)
    threshold = contrast_threshold(background)
    mask = gray.point([255 if abs(value - background) >= threshold else 0
                       for value in range(256)])
    pixels = mask.load()
    obstacles = []
    start = last = None
    top, bottom, count = image.height, 0, 0

    def save_obstacle():
        if start is not None and last - start + 1 >= 5 and count >= 14:
            # 偵測區底部在地面線上方約 3 像素；恐龍腳底約在地面線下方 7 像素。
            bottom_above_ground = image.height - 1 - bottom + 10
            if bottom_above_ground < 48:
                height_above_ground = image.height + 9 - top
                obstacles.append((start, last, bottom_above_ground, height_above_ground))

    for column in range(image.width):
        ys = [row for row in range(image.height) if pixels[column, row]]
        if len(ys) < 2:
            continue
        if start is not None and column - last > 10:
            save_obstacle()
            start = None
        if start is None:
            start, top, bottom, count = column, image.height, 0, 0
        last = column
        top = min(top, ys[0])
        bottom = max(bottom, ys[-1])
        count += len(ys)
    save_obstacle()
    return obstacles

def preview_position(region, screen_size, window_size):
    """選擇不會遮住截圖偵測區的螢幕角落。"""
    screen_width, screen_height = screen_size
    window_width, window_height = window_size
    if screen_width < window_width + 24 or screen_height < window_height + 24:
        return None
    left, top, width, height = region
    for x, y in ((screen_width - window_width - 12, screen_height - window_height - 12),
                 (12, screen_height - window_height - 12),
                 (screen_width - window_width - 12, 12), (12, 12)):
        if (x + window_width + 8 <= left or x >= left + width + 8 or
                y + window_height + 8 <= top or y >= top + height + 8):
            return x, y
    return None

def annotate_screenshot(image, obstacles, jump_boundary):
    """在截圖上標記障礙物；不修改用來判斷的原始畫面。"""
    annotated = image.convert('RGB').copy()
    draw = ImageDraw.Draw(annotated)
    boundary = max(0, min(image.width - 1, round(jump_boundary)))
    draw.line((boundary, 0, boundary, image.height - 1), fill='#00cfff', width=2)
    for index, (left, right, bottom_height, top_height) in enumerate(obstacles):
        top = max(0, image.height + 9 - top_height)
        bottom = min(image.height - 1, image.height + 9 - bottom_height)
        color = '#ff4444' if index == 0 else '#ffcc00'
        draw.rectangle((left, top, right, bottom), outline=color, width=2)
    return annotated

class ScreenshotPreview:
    def __init__(self, region, screen_size):
        self.root = None
        self.last_update = float('-inf')
        window_size = (min(480, screen_size[0] - 24), 165)
        position = preview_position(region, screen_size, window_size)
        if position is None:
            print('螢幕空間不足，無法在不遮住偵測區的情況下顯示預覽。')
            return
        try:
            self.root = tk.Tk()
            self.root.overrideredirect(True)  # 固定位置，避免拖曳後遮住偵測區。
            self.root.attributes('-topmost', True)
            self.root.geometry(f'{window_size[0]}x{window_size[1]}+{position[0]}+{position[1]}')
            self.root.configure(bg='#20242c')
            header = tk.Frame(self.root, bg='#20242c')
            header.pack(fill='x')
            tk.Label(header, text='紅框：障礙物｜藍線：起跳範圍', fg='white',
                     bg='#20242c').pack(side='left', padx=8)
            tk.Button(header, text='關閉', command=self.close, takefocus=False).pack(side='right')
            self.image_label = tk.Label(self.root, bg='#20242c')
            self.image_label.pack()
            self.status_label = tk.Label(self.root, text='等待畫面…', fg='white', bg='#20242c')
            self.status_label.pack()
            self.window_width = window_size[0]
            self.root.update()
            prevent_preview_activation(self.root)
        except tk.TclError as error:
            self.close()
            print(f'無法開啟截圖預覽：{error}')

    def show(self, image, obstacles, status, now, jump_boundary):
        if self.root is None or now - self.last_update < 0.25:
            return
        try:
            annotated = annotate_screenshot(image, obstacles, jump_boundary)
            annotated.thumbnail((self.window_width - 16, 90), Image.Resampling.NEAREST)
            self.photo = ImageTk.PhotoImage(annotated, master=self.root)
            self.image_label.configure(image=self.photo)
            self.status_label.configure(text=status)
            self.root.update()
            self.last_update = now
        except tk.TclError:
            self.close()

    def close(self):
        if self.root is not None:
            try:
                self.root.destroy()
            except tk.TclError:
                pass
            self.root = None

class SpeedEstimator:
    """用連續截圖估算速度；估算失敗也不影響近距離跳躍。"""
    def __init__(self):
        self.x = None
        self.seen_at = None
        self.speed = 360.0

    def update(self, x, now):
        if x is None:
            if self.seen_at is not None and now - self.seen_at > 1.0:
                self.x = self.seen_at = None
            return

        if self.x is None or now - self.seen_at > 1.0 or x > self.x + 5:
            self.x, self.seen_at = x, now
            return

        elapsed = now - self.seen_at
        distance = self.x - x
        if elapsed >= 0.02 and distance >= 4:
            measured = distance / elapsed
            if 120 <= measured <= 1200:
                self.speed = 0.4 * measured + 0.6 * self.speed
                self.x, self.seen_at = x, now
            elif measured > 1200:
                self.x, self.seen_at = x, now

def play_dino():
    old_failsafe, old_pause = pyg.FAILSAFE, pyg.PAUSE
    pyg.FAILSAFE, pyg.PAUSE = True, 0
    ducking = False
    preview = None
    try:
        print('在 GAME OVER 畫面，8 秒內用 Alt + Tab 切回 Chrome，')
        print('將滑鼠移到恐龍頭部右邊、靠近水平地面線的位置，保持不動。')
        time.sleep(8)
        pyg.failSafeCheck()
        x, ground_y = pyg.position()
        screen_width, screen_height = pyg.size()
        # chrome://dino/ 遊戲開始後會將 600 像素的畫布放大至視窗寬度。
        scale = screen_width / 600
        sample_left = x + round(22 * scale)
        sample_width = min(round(160 * scale), screen_width - sample_left)
        sample_top = max(0, ground_y - round(30 * scale))
        sample_bottom = min(screen_height, ground_y + round(30 * scale))
        if sample_width < 60 * scale or sample_bottom <= sample_top:
            raise ValueError('滑鼠位置太靠近螢幕邊緣，請重新選擇恐龍前方的地面線。')
        sample = pyg.screenshot(region=(sample_left, sample_top, sample_width,
                                        sample_bottom - sample_top))
        sample = sample.resize((round(sample_width / scale),
                                round((sample_bottom - sample_top) / scale)),
                               Image.Resampling.NEAREST)
        line = find_ground_line(sample, (ground_y - sample_top) / scale)
        if line is None:
            raise ValueError('找不到地面線；請確認 Chrome 遊戲畫面可見，並把滑鼠移到恐龍前方靠近地面的位置。')
        ground_y = sample_top + round(line * scale)
        left = x + round(8 * scale)
        width = min(round(SCAN_WIDTH * scale), screen_width - left)
        height = round(SCAN_HEIGHT * scale)
        top = ground_y - round(75 * scale)
        if top < 0 or width < 100 * scale:
            raise ValueError('偵測位置太靠近螢幕邊緣；請重新選擇恐龍右側的地面線位置。')
        region = (left, top, width, height)
        normalized_width = round(width / scale)
        if SHOW_PREVIEW:
            avoid_region = (x, top, left + width - x, height)
            preview = ScreenshotPreview(avoid_region, (screen_width, screen_height))
        print(f'偵測區域：{region}，比例：約 {scale:.2f}。正在重新開始。')
        pyg.click(x + round(3 * scale), ground_y - round(20 * scale))
        game_window = foreground_window()
        started = time.monotonic()
        jump_ready_at = started
        speed_estimator = SpeedEstimator()
        duck_until = 0.0
        previous_x = None
        jump_armed = True
        jump_notice_until = 0.0
        focus_warning_shown = False

        while True:
            pyg.failSafeCheck()
            capture_started = time.monotonic()
            image = pyg.screenshot(region=region)
            captured = time.monotonic()
            frame_time = (capture_started + captured) / 2
            image = image.resize((normalized_width, SCAN_HEIGHT), Image.Resampling.NEAREST)
            obstacles = find_obstacles(image)
            now = time.monotonic()
            status = '等待障礙物'
            jump_threshold = JUMP_DISTANCE + max(0, speed_estimator.speed - 360) * 0.16
            jump_boundary = jump_threshold - 8 + speed_estimator.speed * (now - frame_time)
            if game_window is not None and foreground_window() != game_window:
                status = 'Chrome 未取得鍵盤焦點，請點一下遊戲畫面'
                if not focus_warning_shown:
                    print(status)
                    focus_warning_shown = True
                if ducking:
                    pyg.keyUp('down')
                    ducking = False
                if preview is not None:
                    preview.show(image, obstacles, status, now, jump_boundary)
                time.sleep(0.05)
                continue
            focus_warning_shown = False
            if ducking and now >= duck_until:
                pyg.keyUp('down')
                ducking = False

            if obstacles:
                obstacle_x, obstacle_right, bottom_height, _ = obstacles[0]
                speed_estimator.update(obstacle_x, frame_time)
                speed = speed_estimator.speed
                obstacle_width = obstacle_right - obstacle_x + 1
                # 速度只用來補償擷取延遲及加速；是否跳躍由截圖中的距離決定。
                jump_threshold = JUMP_DISTANCE + max(0, speed - 360) * 0.16
                capture_shift = speed * (now - frame_time)
                distance = max(0, obstacle_x + 8 - capture_shift)
                jump_boundary = jump_threshold - 8 + capture_shift
                if previous_x is not None and obstacle_x > previous_x + 15:
                    jump_armed = True
                previous_x = obstacle_x
                if distance > jump_threshold + 20:
                    jump_armed = True
                status = f'距離 {distance:.0f}，起跳範圍 {jump_threshold:.0f}'

                if ducking:
                    # 翼龍拍翅可能讓高度判斷短暫改變，先保持蹲下直到通過。
                    if bottom_height >= 27:
                        duck_until = max(duck_until, now +
                                         (distance + obstacle_width + 38) / speed + 0.1)
                elif bottom_height >= 27:
                    if distance <= max(jump_threshold, speed * 0.30):
                        pyg.keyDown('down')
                        ducking = True
                        duck_until = max(duck_until, now +
                                         (distance + obstacle_width + 38) / speed + 0.1)
                elif distance <= jump_threshold and jump_armed:
                    if now >= jump_ready_at:
                        if ducking:
                            pyg.keyUp('down')
                            ducking = False
                        pyg.press('up')
                        jump_ready_at = time.monotonic() + JUMP_COOLDOWN
                        jump_notice_until = time.monotonic() + 0.5
                        jump_armed = False
                    else:
                        status = '障礙物已接近，等待落地後跳躍'
                elif not jump_armed:
                    status = '已跳躍，等待障礙物通過'
            else:
                speed_estimator.update(None, frame_time)
                previous_x = None
                jump_armed = True

            now = time.monotonic()
            if ducking and now >= duck_until:
                pyg.keyUp('down')
                ducking = False
            if ducking:
                status = '向下鍵：蹲下'
            elif now < jump_notice_until:
                status = '向上鍵：跳躍'
            if preview is not None:
                preview.show(image, obstacles, status, now, jump_boundary)
            time.sleep(0.01)
    except (KeyboardInterrupt, pyg.FailSafeException):
        print('已停止自動遊戲。')
    finally:
        if preview is not None:
            preview.close()
        # 緊急停止時也要放開程式按住的向下鍵。
        if ducking:
            pyg.FAILSAFE = False
            pyg.keyUp('down')
        pyg.FAILSAFE, pyg.PAUSE = old_failsafe, old_pause

play_dino()